# **Taller II: Preparación de Datos para la Implementación de Modelos de Predicción y Optimización del Consumo Energético**

---
**Institución:** Universidad Pedagógica y Tecnológica de Colombia (UPTC)  
**Programa:** Especialización en Inteligencia Artificial  
**Asignatura:** Preparación y Análisis de Datos para Inteligencia Artificial  
**Caso de Estudio:** Gestión y Eficiencia Energética Multisede UPTC (2018 - 2025)  
**Entorno de Ejecución:** Python 3 / Google Colab / Jupyter Notebook  
---

## **1. Propósito de la Actividad**

Diseñar y ejecutar un **proceso integral de preparación de datos** que permita transformar registros históricos de consumo energético en un conjunto de datos limpio, consistente y estructurado, apto para la implementación de modelos de aprendizaje automático (*Machine Learning*) orientados a la predicción del consumo energético y al apoyo de estrategias de optimización y sostenibilidad institucional en la UPTC.

> [!NOTE]
> En la Ciencia de Datos e Inteligencia Artificial aplicada a la ingeniería y gestión de recursos energéticos, la fase de preparación de datos representa típicamente entre el 70% y el 80% del esfuerzo total de un proyecto. Garantizar la calidad, consistencia física y ausencia de fuga de información (*data leakage*) es un prerrequisito indispensable para que los algoritmos predictivos aprendan patrones reales y generalizables.

## **2. Contexto del Problema**

La **Universidad Pedagógica y Tecnológica de Colombia (UPTC)** desarrolla sus actividades académicas, investigativas y administrativas en cuatro sedes principales ubicadas en el departamento de Boyacá, caracterizadas por poblaciones estudiantiles y dinámicas operativas diferenciadas:

| Sede | Población Estudiantil | Perfil Operativo y Dinámica Principal |
| :--- | :---: | :--- |
| **Tunja (Sede Central)** | ~18.000 estudiantes | Sede principal, alta concentración de facultades, rectoría y laboratorios de alta demanda. |
| **Duitama (Facultad Seccional)** | ~5.500 estudiantes | Sede con fuerte perfil técnico-industrial, ingeniería electromecánica y laboratorios especializados. |
| **Sogamoso (Facultad Seccional)** | ~6.000 estudiantes | Sede con programas minero-metalúrgicos, geología, sistemas y talleres pesados. |
| **Chiquinquirá (Facultad Seccional)** | ~2.000 estudiantes | Sede enfocada en ciencias económicas, administrativas y programas de educación. |

### Sectores Institucionales de Consumo
El funcionamiento de cada sede implica consumos energéticos distribuidos en cinco sectores clave:
1. **Comedores:** Demanda intensiva en franjas horarias puntuales (desayuno, almuerzo, refrigerio), uso de equipos de cocción, refrigeración y lavado industrial.
2. **Salones:** Consumo regido por horarios de clase diurnos y nocturnos (iluminación, equipos audiovisuales y ventilación).
3. **Laboratorios:** Consumo elevado y continuo asociado a maquinaria especializada, bancos de prueba, servidores de cómputo, autoclaves y sistemas de acondicionamiento continuo.
4. **Auditorios:** Demanda esporádica pero con picos concentrados en eventos institucionales, congresos y graduaciones (iluminación de potencia, amplificación y climatización).
5. **Oficinas:** Consumo administrativo estable durante jornada laboral de lunes a viernes (cómputo personal, fotocopiado e iluminación).

### Diagnóstico Inicial de Calidad del Dataset
Se dispone del archivo histórico `consumos_uptc.csv` con lecturas horarias recolectadas entre 2018 y 2025. Una auditoría preliminar ha evidenciado los siguientes problemas de calidad de datos:
- **Valores faltantes (nulos):** Provocados por caídas en la telemetría, pérdida de paquetes o fallas en sensores IoT.
- **Registros duplicados:** Producidos por retransmisiones automáticas del protocolo de comunicación o desajustes de reloj.
- **Inconsistencias categóricas:** Múltiples convenciones tipográficas en identificadores (`UPTC_DUI`, `UPTC-DUI`, `uptc_dui`) y periodos académicos.
- **Valores negativos:** Registros físicamente imposibles en variables de consumo energético y huella de carbono.
- **Valores atípicos (outliers):** Mediciones anómalas que requieren evaluación según el dominio de la ingeniería energética.

## **3. Objetivos Específicos**

Para dar cumplimiento al propósito general, se establecen seis objetivos metodológicos estructurados:

1. **Realizar la exploración inicial** del conjunto de datos para identificar su estructura, dimensiones, tipos de variables y posibles problemas de calidad.
2. **Aplicar técnicas de limpieza y depuración** de datos mediante el tratamiento de valores faltantes, detección de registros inconsistentes, eliminación de duplicados y corrección de anomalías en variables de consumo.
3. **Analizar estadísticamente las variables energéticas** para identificar patrones de comportamiento, distribuciones y relaciones entre sectores de consumo.
4. **Implementar procesos de transformación y preparación** de variables que permitan generar un conjunto de datos adecuado para técnicas de modelado predictivo.
5. **Seleccionar las variables objetivo** y definir el conjunto de características (*features*) relevantes para la construcción de modelos de predicción del consumo energético, evitando fuga de datos.
6. **Construir un conjunto de datos final** que garantice calidad, integridad y consistencia para etapas posteriores de entrenamiento, validación e interpretación de modelos de Inteligencia Artificial.

---
## **0. Configuración del Entorno de Trabajo e Importación de Librerías**

Importamos las librerías científicas fundamentales del ecosistema Python para análisis de datos, visualización y preparación matricial.

In [ ]:
# Importación de librerías esenciales para Ciencia de Datos
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de advertencias y precisión visual
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 35)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

# Configuración estética para gráficos científicos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 110

print('Librerías importadas correctamente.')
print(f'Versión de Pandas: {pd.__version__}')
print(f'Versión de NumPy : {np.__version__}')

: 

---
# **Fase I. Carga y Exploración Inicial de Datos**

La Fase I se enfoca en el reconocimiento preliminar de la estructura del dataset, identificando dimensiones, tipos de datos, distribución multisede y síntomas iniciales de anomalías.

### **4.1.1. Paso 1: Cargar los datos desde Google Drive a Google Colab**

De acuerdo con las especificaciones del taller, el dataset debe cargarse en el entorno de ejecución. Para garantizar máxima portabilidad tanto en Google Colab como en entornos locales (Jupyter Lab, VS Code), el siguiente bloque implementa una estrategia modular:

1. **Opción A (Recomendada en Colab con Drive):** Monta Google Drive mediante `from google.colab import drive; drive.mount('/content/drive')` y lee directamente desde tu unidad de almacenamiento persistente.
2. **Opción B (Carga interactiva):** Permite subir el archivo `consumos_uptc.csv` directamente desde tu computador mediante el selector `files.upload()`.
3. **Opción C (Detección local automática):** Si el cuaderno se ejecuta localmente o el archivo ya se encuentra en el directorio de trabajo, lo detecta y lo carga de inmediato sin intervención manual.

In [ ]:
# Implementación modular para carga de datos (Google Drive / Upload / Directorio Local)
nombre_archivo = None

# 1. Verificar si el archivo ya existe localmente o con nombres comunes en la carpeta de trabajo
candidatos = [
    'consumos_uptc.csv',
    'consumos_uptc (1).csv',
    '/content/consumos_uptc.csv',
    '/content/drive/MyDrive/consumos_uptc.csv',
    '/content/drive/MyDrive/Clase3/consumos_uptc.csv'
]

for ruta in candidatos:
    if os.path.exists(ruta):
        nombre_archivo = ruta
        print(f'✓ Archivo encontrado localmente en: {nombre_archivo}')
        break

# 2. Si no se encuentra localmente, verificar si estamos en Google Colab para ofrecer Drive o Upload
if nombre_archivo is None:
    try:
        from google.colab import drive
        print('Intentando montar Google Drive...')
        drive.mount('/content/drive')
        ruta_drive = '/content/drive/MyDrive/consumos_uptc.csv'
        if os.path.exists(ruta_drive):
            nombre_archivo = ruta_drive
            print(f'✓ Archivo cargado desde Google Drive: {nombre_archivo}')
        else:
            from google.colab import files
            print('Por favor, selecciona el archivo consumos_uptc.csv desde tu equipo:')
            uploaded = files.upload()
            nombre_archivo = list(uploaded.keys())[0]
    except Exception as e:
        raise FileNotFoundError(
            'No se encontró el archivo consumos_uptc.csv. Colócalo en el directorio o móntalo en Google Drive.'
        )

# Cargar en un DataFrame principal
df = pd.read_csv(nombre_archivo)
print(f'\n-> Dataset cargado con éxito en memoria. Registros leídos: {len(df):,}')

### **4.1.2. Paso 2: Identificar dimensiones del conjunto de datos**

Examinamos la dimensionalidad del conjunto de datos a través del atributo `.shape`, calculando el número de filas (observaciones temporales horarias), columnas (variables o sensores) y el volumen total de datos generados.

In [ ]:
filas, columnas = df.shape
total_celdas = filas * columnas
memoria_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

print(f'=== Dimensiones del Conjunto de Datos ===')
print(f'Total de registros (filas)       : {filas:,}')
print(f'Total de variables (columnas)    : {columnas}')
print(f'Total de celdas de datos         : {total_celdas:,}')
print(f'Consumo aproximado de memoria RAM: {memoria_mb:.2f} MB')

> [!NOTE]
> **Interpretación:** El conjunto de datos posee **275.387 filas y 26 columnas**, lo que representa un universo de más de **7,16 millones de puntos de datos**. Esta granularidad horaria durante un periodo de 7 años (2018 a 2025) proporciona una base muestral robusta pero exige verificar consistencia y optimizar los tipos de datos.

In [ ]:
# Visualización de los primeros 5 registros para inspección de estructura
print('Primeras 5 observaciones del dataset:')
display(df.head())

In [ ]:
# Visualización de los últimos 5 registros
print('Últimas 5 observaciones del dataset:')
display(df.tail())

### **4.1.3. Paso 3: Analizar variables y tipos de datos**

Utilizamos `df.info()` y una taxonomía estructurada para comprender el rol de cada una de las 26 variables del dataset. Es crucial distinguir entre variables cuantitativas (continuas y discretas), categóricas (nominales y ordinales) y de fecha/hora.

In [ ]:
# Diagnóstico de tipos de datos técnicos en Pandas
df.info()

#### **Taxonomía Semántica de las Variables**

A partir de la inspección técnica, clasificamos las 26 variables en 5 dimensiones operativas:

| Dimensión | Variables | Tipo Nativo | Rol en el Análisis |
| :--- | :--- | :---: | :--- |
| **1. Identificación y Tiempo** | `reading_id`, `timestamp`, `hora`, `dia_semana`, `dia_nombre`, `mes`, `trimestre`, `año` | `object`, `int64` | Identificadores y descomposición temporal del ciclo circadiano y estacional. |
| **2. Espacial y Contexto** | `sede`, `sede_id`, `periodo_academico` | `object` | Identificadores geográficos y estado del calendario institucional. |
| **3. Banderas de Calendario** | `es_fin_semana`, `es_festivo`, `es_semana_parciales`, `es_semana_finales` | `bool` | Factores exógenos que modulan drásticamente la asistencia y demanda. |
| **4. Consumo por Sectores** | `energia_comedor_kwh`, `energia_salones_kwh`, `energia_laboratorios_kwh`, `energia_auditorios_kwh`, `energia_oficinas_kwh` | `float64` | Medición subsectorial de energía eléctrica activa por dependencia. |
| **5. Magnitudes Agregadas y Ambientales** | `energia_total_kwh`, `potencia_total_kw`, `agua_litros`, `temperatura_exterior_c`, `ocupacion_pct`, `co2_kg` | `float64` | Totalizador general, demanda instantánea, variables ambientales e impacto ambiental. |

> [!WARNING]
> **Hallazgo Crítico:** La columna `timestamp` fue leída como tipo `object` (cadena de texto). Para análisis de series temporales y validación cronológica, debe ser transformada a formato datetime estándar ISO-8601.

### **4.1.4. Paso 4: Explorar valores únicos por sede**

Analizamos la distribución de observaciones por sede para comprobar si el muestreo es equilibrado y si existen inconsistencias en las variables geográficas y administrativas (`sede`, `sede_id`, `periodo_academico`).

In [ ]:
print('--- Distribución de Registros por Sede (Nombre) ---')
sede_dist = pd.DataFrame({
    'Registros': df['sede'].value_counts(dropna=False),
    'Porcentaje (%)': (df['sede'].value_counts(dropna=False, normalize=True) * 100).round(2)
})
display(sede_dist)

print('\n--- Valores Únicos en Sede ID ---')
print(f'Número de categorías en sede_id: {df["sede_id"].nunique()} (Se esperaban exactamente 4)')
display(df['sede_id'].value_counts())

print('\n--- Valores Únicos en Periodo Académico ---')
print(f'Número de categorías en periodo_academico: {df["periodo_academico"].nunique()}')
display(df['periodo_academico'].value_counts())

> [!NOTE]
> **Interpretación:**
> 1. **Balance temporal multisede:** Cada una de las cuatro sedes tiene un número prácticamente idéntico de registros (~68.800 filas cada una, ~25% del total). Esto confirma que los sensores registraron el mismo intervalo histórico para todas las sedes, independientemente de la diferencia en su población estudiantil (Tunja con 18.000 vs Chiquinquirá con 2.000).
> 2. **Inconsistencias en `sede_id`:** Se encontraron **12 categorías distintas** cuando deberían existir únicamente 4. Aparecen variantes de escritura para la misma sede, por ejemplo: `UPTC_DUI`, `UPTC-DUI` y `uptc_dui`. Este error proviene de variaciones en la codificación de los microcontroladores o configuración de gateways IoT.
> 3. **Inconsistencias en `periodo_academico`:** Aparecen etiquetas dispersas como `semestre_1`, `SEMESTRE_1`, `Semestre_1` y `semestre1`, así como `vacaciones` sin especificar si corresponden a mitad o fin de año.

### **4.1.5. Paso 5: Obtener estadísticas descriptivas**

Calculamos las medidas de tendencia central, dispersión, posición y valores extremos para todas las variables numéricas del dataset mediante `df.describe().T`.

In [ ]:
# Resumen estadístico descriptivo de variables cuantitativas
desc_stats = df.describe().T
desc_stats['iqr'] = desc_stats['75%'] - desc_stats['25%']
desc_stats['range'] = desc_stats['max'] - desc_stats['min']
display(desc_stats[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'iqr']])

> [!IMPORTANT]
> **Observaciones Clave de las Estadísticas Descriptivas:**
> - **Valores Mínimos Negativos:** `energia_total_kwh` presenta un mínimo negativo de **-2.820 kWh** y `co2_kg` presenta un mínimo de **-1.466 kg**. El consumo de energía de una instalación consumidora y las emisiones de CO2 asociadas no pueden ser inferiores a cero bajo las leyes de la termodinámica.
> - **Valores Faltantes (`count`):** Varias variables numéricas tienen un conteo inferior a 275.387, lo que certifica la presencia de valores nulos.
> - **Magnitudes Extremas (Outliers):** La variable `agua_litros` alcanza máximos muy por encima del percentil 75, lo que sugiere fugas de tubería o picos puntuales de consumo.
> - **Asimetría Positiva:** En las variables de consumo por sector, la media es sistemáticamente superior a la mediana (50%), lo que denota distribuciones con cola pesada hacia la derecha (consumo bajo la mayor parte del tiempo con picos durante horarios académicos).

### **4.1.6. Paso 6: Detectar posibles inconsistencias**

Sistematizamos la auditoría de calidad de datos mediante la detección algorítmica de todas las anomalías presentes en el dataset crudo antes de iniciar los procedimientos de limpieza.

In [ ]:
# Matriz de auditoría de anomalías e inconsistencias detectadas
nulos_totales = df.isnull().sum()
cols_con_nulos = nulos_totales[nulos_totales > 0]

negativos_totales = {}
cols_numericas = df.select_dtypes(include=[np.number]).columns.drop('reading_id', errors='ignore')
for c in cols_numericas:
    neg_count = (df[c] < 0).sum()
    if neg_count > 0:
        negativos_totales[c] = neg_count

print('=== REPORTE DE INCONSISTENCIAS IDENTIFICADAS EN FASE I ===')
print(f'1. Variables con valores nulos              : {len(cols_con_nulos)}')
for col, val in cols_con_nulos.items():
    print(f'   - {col:25s}: {val:6,d} nulos ({val/len(df)*100:5.2f}%)')

print(f'\n2. Variables con valores negativos (< 0)    : {len(negativos_totales)}')
for col, val in negativos_totales.items():
    print(f'   - {col:25s}: {val:6,d} registros negativos')

print(f'\n3. Dispersión tipográfica en sede_id        : {df["sede_id"].nunique()} variantes (esperadas: 4)')
print(f'4. Dispersión tipográfica en periodo_academ : {df["periodo_academico"].nunique()} variantes (esperadas: 4)')
print(f'5. Duplicados exactos ignorando reading_id  : {df.drop(columns=["reading_id"]).duplicated().sum():,}')
print(f'6. Duplicados por timestamp + sede          : {df.duplicated(subset=["timestamp", "sede"]).sum():,}')

| Inconsistencia Detectada | Diagnóstico Técnico | Impacto en Modelos de IA | Estrategia de Mitigación (Fase II) |
| :--- | :--- | :--- | :--- |
| **Nombres categóricos inconsistentes** | Formatos mixtos en mayúsculas/minúsculas y guiones. | Fragmenta una misma sede o periodo en categorías falsas durante el *One-Hot Encoding*. | Estandarización de cadenas con expresiones vectorizadas. |
| **Valores negativos** | Lecturas < 0 en energía y CO2. | Viola restricciones físicas del dominio energético y distorsiona el cálculo de pérdidas. | Eliminación de registros físicamente imposibles. |
| **Duplicados temporales** | Múltiples lecturas para la misma hora y sede. | Causa sobreajuste (*overfitting*) y ponderación excesiva de eventos duplicados. | Desduplicación conservando la primera medición válida. |
| **Datos faltantes** | Nulos en consumo, clima y ocupación (hasta 12%). | Impide el entrenamiento matricial de la mayoría de algoritmos predictivos. | Imputación robusta con mediana (numéricas) y moda (categóricas). |

---
# **Fase II. Limpieza y Preparación Inicial**

En esta fase ejecutamos la depuración del conjunto de datos aplicando filtros de selección geográfica, saneamiento tipográfico, eliminación de registros físicamente inválidos y desduplicación temporal.

### **Estandarización Previa de Variables Categóricas**

Antes de aplicar el filtrado por sede, homogeneizamos las cadenas de texto en `sede_id` y `periodo_academico`. Esta práctica previene la pérdida accidental de datos por diferencias de mayúsculas, minúsculas o caracteres especiales.

In [ ]:
# 1. Estandarización de sede_id a mayúsculas con guion bajo
df['sede_id'] = df['sede_id'].astype(str).str.upper().str.replace('-', '_').str.strip()

# 2. Estandarización de periodo_academico: minúsculas, sin espacios y unificación de sinónimos
df['periodo_academico'] = (
    df['periodo_academico'].astype(str).str.lower().str.strip()
    .replace({
        'semestre1': 'semestre_1',
        'vacaciones': 'vacaciones_fin'
    })
)

print('Distribución final y limpia de sede_id:')
display(df['sede_id'].value_counts())

print('\nDistribución final y limpia de periodo_academico:')
display(df['periodo_academico'].value_counts())

### **4.2.1. Paso 7: Identificar valores nulos**

Evaluamos la presencia de valores nulos a nivel global en el dataset y visualizamos el porcentaje de ausencia por columna para dimensionar el alcance de la falta de información.

In [ ]:
# Cálculo cuantitativo de valores faltantes
conteo_nulos = df.isnull().sum()
pct_nulos = (conteo_nulos / len(df) * 100).round(2)
tabla_nulos = pd.DataFrame({'Nulos': conteo_nulos, 'Porcentaje (%)': pct_nulos})
tabla_nulos = tabla_nulos[tabla_nulos['Nulos'] > 0].sort_values(by='Porcentaje (%)', ascending=False)

print('Variables con presencia de valores nulos:')
display(tabla_nulos)

# Visualización del porcentaje de valores nulos
plt.figure(figsize=(9, 4))
sns.barplot(x=tabla_nulos['Porcentaje (%)'], y=tabla_nulos.index, palette='Blues_r')
plt.title('Porcentaje de Valores Nulos por Variable (Dataset General)', fontsize=12, fontweight='bold')
plt.xlabel('Porcentaje Faltante (%)', fontsize=10)
plt.ylabel('Variable', fontsize=10)
plt.xlim(0, 15)
for i, v in enumerate(tabla_nulos['Porcentaje (%)']):
    plt.text(v + 0.2, i, f'{v:.2f}%', va='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

> [!NOTE]
> **Interpretación:** La variable con mayor proporción de datos faltantes es `ocupacion_pct` (~12,1%), seguida de `temperatura_exterior_c` (~6,0%) y `energia_auditorios_kwh` (~4,9%). Como ninguna variable supera el umbral crítico del 20-30%, no es necesario descartar columnas completas; en su lugar, se realizará una imputación estadística en la Fase III.

### **4.2.2. Paso 8: Filtrar registros correspondientes a la sede DUITAMA**

Siguiendo estrictamente las especificaciones del taller (Paso 8), enfocamos el estudio en la **Sede Duitama**, caracterizada por su alta actividad técnica y laboratorios de ingeniería electromecánica.

In [ ]:
# Registro de dimensiones previas al filtrado
registros_antes = len(df)

# Filtrado exacto de la sede Duitama garantizando robustez ante espacios o mayúsculas
df_duitama = df[df['sede'].astype(str).str.strip().str.upper() == 'DUITAMA'].copy()
df = df_duitama  # Asignamos al DataFrame principal de trabajo

registros_despues = len(df)

### **4.2.3. Paso 9: Comparar cantidad de registros antes y después del filtrado**

Construimos una tabla comparativa y métricas de retención para validar el resultado del filtro geográfico.

In [ ]:
registros_descartados = registros_antes - registros_despues
pct_retenido = (registros_despues / registros_antes) * 100
pct_descartado = (registros_descartados / registros_antes) * 100

resumen_filtrado = pd.DataFrame({
    'Etapa': ['Dataset Completo (4 Sedes)', 'Sede DUITAMA (Filtrado)', 'Sedes Excluidas (Tunja, Sogamoso, Chiquinquirá)'],
    'Cantidad de Filas': [registros_antes, registros_despues, registros_descartados],
    'Porcentaje (%)': [100.0, pct_retenido, pct_descartado]
})
display(resumen_filtrado)

print(f'-> Registros antes del filtrado   : {registros_antes:,}')
print(f'-> Registros después del filtrado : {registros_despues:,}')
print(f'-> Registros descartados          : {registros_descartados:,} ({pct_descartado:.2f}%)')

> [!NOTE]
> **Interpretación:** Al segmentar por Duitama se retienen **68.862 registros**, lo que corresponde exactamente al **25,01%** del conjunto general. Esto es coherente con el balance del sistema de medición (4 sedes monitoreadas con igual frecuencia temporal horaria). Disponer de más de 68.000 observaciones horarias garantiza una muestra de gran tamaño para el modelado predictivo.

### **4.2.4. Paso 10: Detectar valores negativos en variables clave**

De acuerdo con las instrucciones explícitas del taller (Paso 10), auditamos la presencia de valores negativos (< 0) en las siguientes 7 variables fundamentales, además de la energía total:

1. **Energía comedor** (`energia_comedor_kwh`)
2. **Energía salones** (`energia_salones_kwh`)
3. **Energía laboratorios** (`energia_laboratorios_kwh`)
4. **Energía auditorios** (`energia_auditorios_kwh`)
5. **Energía oficinas** (`energia_oficinas_kwh`)
6. **Agua (litros)** (`agua_litros`)
7. **CO2 (kg)** (`co2_kg`)
8. *Energía total* (`energia_total_kwh`)

In [ ]:
# Lista explícita de variables a inspeccionar según el Paso 10
cols_negativas = [
    'energia_comedor_kwh',
    'energia_salones_kwh',
    'energia_laboratorios_kwh',
    'energia_auditorios_kwh',
    'energia_oficinas_kwh',
    'agua_litros',
    'co2_kg',
    'energia_total_kwh'
]

# Verificación exhaustiva de cada variable
analisis_negativos = []
for c in cols_negativas:
    neg_count = (df[c] < 0).sum()
    min_val = df[c].min()
    pct_neg = (neg_count / len(df)) * 100
    estado = 'ANÓMALO (Inválido)' if neg_count > 0 else 'CORRECTO (Válido)'
    analisis_negativos.append({
        'Variable': c,
        'Valores Negativos': neg_count,
        'Porcentaje (%)': round(pct_neg, 3),
        'Valor Mínimo Registrado': round(min_val, 3) if pd.notnull(min_val) else np.nan,
        'Estado Físico': estado
    })

df_analisis_neg = pd.DataFrame(analisis_negativos)
display(df_analisis_neg)

> [!CAUTION]
> **Fundamento Físico de la Inconsistencia:**
> - El consumo eléctrico activo de una edificación receptora conectada a la red convencional no puede ser negativo: las cargas institucionales (iluminación, cómputo, motores) son disipativas de energía (\(P(t) \ge 0\)). No existen sistemas de inyección distribuida fotovoltaica con balance neto inverso en estos medidores que justifiquen lecturas negativas.
> - Asimismo, la huella de carbono asociada al consumo eléctrico (\(CO_2 = \text{kWh} \times \text{factor de emisión}\)) no puede ser inferior a cero.
> - Se observa que **los cinco sectores individuales de energía y el agua no presentan valores negativos (0 anomalías)**. Los valores negativos se concentran exclusivamente en `co2_kg` (143 registros) y `energia_total_kwh` (155 registros), indicando un error en el sensor totalizador o un artefacto computacional de registro.

### **4.2.5. Paso 11: Eliminar registros inválidos**

Procedemos a eliminar los registros con valores negativos. Esta decisión se fundamenta en que los registros corrompidos corresponden a lecturas erróneas donde no es posible inferir el valor verdadero real. Dado que representan únicamente el **0,23%** de la sede Duitama, su descarte no compromete la representatividad muestral.

In [ ]:
# Identificar máscara de filas con al menos un valor negativo en las columnas críticas
filas_negativas = (df[cols_negativas] < 0).any(axis=1)
n_negativas = filas_negativas.sum()

print(f'Total de registros con valores negativos a eliminar: {n_negativas}')
print(f'Registros antes de eliminar negativos: {len(df):,}')

# Filtrar conservando únicamente filas sin negativos
df = df[~filas_negativas].copy()
print(f'Registros después de eliminar negativos: {len(df):,}')

> [!NOTE]
> **Resultado:** Se eliminaron con éxito los **155 registros inválidos**, pasando de 68.862 a **68.707 registros**.

### **4.2.6. Paso 12: Verificar registros duplicados**

Auditamos la existencia de registros duplicados bajo dos ópticas complementarias:
1. **Duplicados exactos completos:** Filas idénticas en todas sus columnas (o ignorando el identificador artificial `reading_id`).
2. **Colisiones temporales (Timestamp + Sede):** Puesto que la red IoT genera una lectura horaria por sede, no pueden existir dos registros para el mismo instante de tiempo en Duitama.

In [ ]:
# 1. Duplicados ignorando el identificador unívoco reading_id
dup_exactos = df.drop(columns=['reading_id'], errors='ignore').duplicated().sum()

# 2. Colisiones por timestamp + sede (criterio operativo de series de tiempo)
dup_temporales = df.duplicated(subset=['timestamp', 'sede']).sum()

print(f'Duplicados exactos (excluyendo reading_id): {dup_exactos}')
print(f'Duplicados por fecha/hora y sede          : {dup_temporales}')

# Inspeccionar ejemplos de colisiones temporales
print('\nEjemplo de registros en colisión temporal (mismo timestamp):')
filas_colision = df[df.duplicated(subset=['timestamp', 'sede'], keep=False)].sort_values('timestamp')
display(filas_colision[['timestamp', 'sede', 'energia_total_kwh', 'energia_laboratorios_kwh', 'co2_kg', 'ocupacion_pct']].head(6))

> [!NOTE]
> **Diagnóstico de los Duplicados:**
> - No existen duplicados exactos en todas las columnas, pero se detectan **220 lecturas redundantes** para la misma fecha y hora.
> - Al inspeccionar los pares duplicados, se observa que los valores de consumo energético por sector y la energía total son **exactamente idénticos**, presentando ligeras discrepancias únicamente en los decimales de `co2_kg`. Esto ocurre típicamente cuando el gateway reenvía el paquete por confirmación tardía (ACK timeout). Por regla de ingeniería de datos, se conserva la primera medición registrada.

### **4.2.7. Paso 13: Eliminar duplicados**

Aplicamos el descarte de duplicados temporales conservando la primera lectura válida (`keep='first'`) y restablecemos el índice de filas.

In [ ]:
registros_pre_dup = len(df)
df = df.drop_duplicates(subset=['timestamp', 'sede'], keep='first').reset_index(drop=True)
registros_post_dup = len(df)
eliminados_dup = registros_pre_dup - registros_post_dup

print(f'Registros antes de eliminar duplicados : {registros_pre_dup:,}')
print(f'Duplicados eliminados                 : {eliminados_dup:,}')
print(f'Registros limpios restantes           : {registros_post_dup:,}')

In [ ]:
# Tabla consolidada de la Fase II de Limpieza
resumen_depuracion = pd.DataFrame({
    'Fase / Etapa': [
        '1. Dataset Original (Consumos UPTC)',
        '2. Filtrado Sede Duitama',
        '3. Depuración de Valores Negativos (< 0)',
        '4. Depuración de Duplicados Temporales'
    ],
    'Registros Resultantes': [registros_antes, registros_despues, len(df) + eliminados_dup, len(df)],
    'Registros Descartados en Etapa': [0, registros_antes - registros_despues, n_negativas, eliminados_dup],
    'Porcentaje Retenido (%)': [
        100.0,
        round(registros_despues / registros_antes * 100, 2),
        round((len(df) + eliminados_dup) / registros_antes * 100, 2),
        round(len(df) / registros_antes * 100, 2)
    ]
})
display(resumen_depuracion)

> [!NOTE]
> **Balance de Limpieza:** El dataset para la Sede Duitama queda depurado con exactamente **68.487 registros**, libre de inconsistencias geográficas, sin registros negativos y sin redundancias temporales.

---
# **Fase III. Análisis Exploratorio de Datos (EDA)**

En la Fase III realizamos la caracterización estadística, distribución probabilística, tratamiento formal de datos faltantes y análisis de correlación entre los sectores de consumo y las variables operacionales.

### **4.3.1. Paso 14: Obtener estadísticas descriptivas de las variables energéticas**

Conforme al Paso 14 del taller, calculamos y comparamos en detalle los estadísticos descriptivos para los **cinco sectores energéticos institucionales**:

- `energia_comedor_kwh`
- `energia_salones_kwh`
- `energia_laboratorios_kwh`
- `energia_auditorios_kwh`
- `energia_oficinas_kwh`

In [ ]:
# Lista de variables energéticas sectoriales
vars_energia = [
    'energia_comedor_kwh',
    'energia_salones_kwh',
    'energia_laboratorios_kwh',
    'energia_auditorios_kwh',
    'energia_oficinas_kwh'
]

# Cálculo de estadísticas completas incluyendo cuartiles, rango intercuartílico, asimetría y curtosis
eda_stats = df[vars_energia].describe().T
eda_stats['mediana'] = df[vars_energia].median()
eda_stats['iqr'] = eda_stats['75%'] - eda_stats['25%']
eda_stats['asimetria (skew)'] = df[vars_energia].skew()
eda_stats['curtosis'] = df[vars_energia].kurtosis()

columnas_ordenadas = ['count', 'mean', 'std', 'min', '25%', 'mediana', '75%', 'max', 'iqr', 'asimetria (skew)', 'curtosis']
display(eda_stats[columnas_ordenadas].round(3))

#### **Análisis Comparativo por Sector Institucional**

1. **Laboratorios (`energia_laboratorios_kwh`):**
   - Es el sector con **mayor consumo medio horario (~3,59 kWh)** y la mayor desviación estándar (~2,14 kWh). Representa el núcleo energético de la sede Duitama, impulsado por maquinaria de talleres, motores, bancos de ensayo y computadores de alto rendimiento.
2. **Oficinas y Salones (`energia_oficinas_kwh`, `energia_salones_kwh`):**
   - Presentan consumos medios intermedios (**~2,58 kWh y ~2,36 kWh**, respectivamente). Sus patrones están fuertemente condicionados por las franjas horarias diurnas y los días laborables de lunes a viernes.
3. **Auditorios y Comedores (`energia_auditorios_kwh`, `energia_comedor_kwh`):**
   - Presentan los consumos basales más reducidos (**medias de ~0,42 kWh y ~0,67 kWh**, con medianas de 0,33 y 0,47 kWh).
   - Sin embargo, exhiben la **mayor asimetría positiva (skewness > 1,3)** y alta curtosis, producto de picos esporádicos pero intensos (franjas de almuerzo de 12:00 a 14:00 y eventos magnos o asambleas en auditorios).
4. **Comportamiento Asimétrico General:**
   - En todos los sectores, la media es significativamente mayor que la mediana (\(\bar{x} > Me\)). Esto demuestra que las distribuciones no son normales, sino fuertemente sesgadas hacia la derecha por la inactividad nocturna y de fin de semana.

### **4.3.2. Paso 15: Visualizar histogramas para cada tipo de variables energéticas**

Generamos visualizaciones de alta calidad técnica para examinar la forma de las distribuciones empíricas de los cinco sectores: histogramas con curva de densidad KDE, diagramas de cajas (*boxplots*) y el perfil horario promedio de consumo.

In [ ]:
# Visualización 1: Histogramas con Densidad Kernel (KDE)
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
paleta = sns.color_palette('tab10', len(vars_energia))

for ax, col, color in zip(axes.flat[:5], vars_energia, paleta):
    nombre_sector = col.replace('energia_', '').replace('_kwh', '').capitalize()
    sns.histplot(df[col].dropna(), bins=45, kde=True, color=color, edgecolor='black', alpha=0.6, ax=ax)
    ax.axvline(df[col].median(), color='red', linestyle='--', linewidth=1.5, label=f'Mediana: {df[col].median():.2f}')
    ax.axvline(df[col].mean(), color='navy', linestyle='-', linewidth=1.5, label=f'Media: {df[col].mean():.2f}')
    ax.set_title(f'Sector: {nombre_sector}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Consumo (kWh)', fontsize=10)
    ax.set_ylabel('Frecuencia', fontsize=10)
    ax.legend(fontsize=9, loc='upper right')

# Ocultar el sexto subplot vacío
axes.flat[5].axis('off')
plt.suptitle('Distribución de Consumo Energético por Sector en Sede Duitama', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Visualización 2: Diagrama de Cajas (Boxplots) comparativo en escala homogénea
plt.figure(figsize=(12, 5))
sns.boxplot(
    data=df[vars_energia],
    orient='h',
    palette='Set2',
    flierprops={'markerfacecolor': 'crimson', 'marker': 'o', 'markersize': 2, 'alpha': 0.5}
)
plt.title('Diagrama de Cajas: Comparación de Dispersión y Outliers por Sector', fontsize=13, fontweight='bold')
plt.xlabel('Consumo Energético (kWh)', fontsize=11)
plt.yticks(
    ticks=range(len(vars_energia)),
    labels=[c.replace('energia_', '').replace('_kwh', '').capitalize() for c in vars_energia],
    fontsize=11
)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Visualización Complementaria: Perfil de Consumo Diario Promedio por Hora (Ciclo Circadiano)
perfil_horario = df.groupby('hora')[vars_energia].mean()

plt.figure(figsize=(13, 5))
for col in vars_energia:
    sector_nombre = col.replace('energia_', '').replace('_kwh', '').capitalize()
    plt.plot(perfil_horario.index, perfil_horario[col], marker='o', linewidth=2, label=sector_nombre)

plt.title('Perfil Horario de Demanda Media por Sector (0:00 a 23:00)', fontsize=13, fontweight='bold')
plt.xlabel('Hora del Día', fontsize=11)
plt.ylabel('Consumo Promedio (kWh)', fontsize=11)
plt.xticks(range(0, 24))
plt.legend(title='Sectores', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

> [!NOTE]
> **Interpretación de las Gráficas:**
> 1. **Concentración cerca de cero (Inflación de ceros / Bimodalidad):** En todos los sectores se observa una gran densidad de lecturas muy bajas correspondientes al horario nocturno (22:00 a 06:00), domingos y periodos de vacaciones intersemestrales.
> 2. **Perfil circadiano coherente:** El consumo se activa a partir de las 06:00, alcanza su cenit entre las 10:00 y las 16:00 (jornada académica y administrativa plena) y decae progresivamente después de las 20:00.
> 3. **Puntos rojos (Outliers en Boxplots):** Los valores atípicos reflejados en el diagrama de cajas corresponden a horas pico de alta demanda académica o eventos especiales reales; **no representan errores de medición**, por lo que deben preservarse para que el modelo predictivo aprenda a captar picos de carga.

### **4.3.3. Paso 16: Imputar valores nulos**

De acuerdo con las instrucciones metodológicas (Paso 16):
- **Mediana** para variables numéricas (cuantitativas).
- **Moda** para variables categóricas (cualitativas).

> [!TIP]
> **Justificación Matemática de la Imputación por Mediana:**
> En distribuciones marcadamente asimétricas con colas derechas y valores atípicos reales, la media aritmética (\(\bar{x}\)) se encuentra sesgada hacia los valores extremos. La mediana (\(Me\)) minimiza el error absoluto medio (norma \(L_1\)) y constituye el estimador de tendencia central más robusto para variables energéticas y meteorológicas.

In [ ]:
# 1. Diagnóstico de valores nulos presentes en Duitama antes de imputar
nulos_pre = df.isnull().sum()
nulos_presentes = nulos_pre[nulos_pre > 0]
print('Valores nulos en Duitama previos a la imputación:')
display(pd.DataFrame({'Nulos': nulos_presentes, '%': (nulos_presentes / len(df) * 100).round(2)}))

# 2. Segmentación de columnas por tipo para estrategia de imputación
cols_num = df.select_dtypes(include=[np.number]).columns.drop('reading_id', errors='ignore')
cols_cat = df.select_dtypes(include=['object', 'string']).columns.drop('timestamp', errors='ignore')

# 3. Aplicación de imputación por Mediana (Numéricas)
valores_imputados_mediana = {}
for c in cols_num:
    if df[c].isnull().sum() > 0:
        med = df[c].median()
        valores_imputados_mediana[c] = med
        df[c] = df[c].fillna(med)

# 4. Aplicación de imputación por Moda (Categóricas)
valores_imputados_moda = {}
for c in cols_cat:
    if df[c].isnull().sum() > 0:
        moda = df[c].mode()[0]
        valores_imputados_moda[c] = moda
        df[c] = df[c].fillna(moda)

# 5. Verificación de ausencia total de nulos
total_nulos_post = df.isnull().sum().sum()
print(f'\n-> Total de valores nulos en el dataset tras la imputación: {total_nulos_post}')
if total_nulos_post == 0:
    print('✓ Verificación exitosa: Dataset completo al 100% sin valores faltantes.')

In [ ]:
# Tabla de valores de imputación utilizados
df_imputaciones = pd.DataFrame([
    {'Variable': col, 'Tipo': 'Numérica', 'Estrategia': 'Mediana', 'Valor Imputado': round(val, 3)}
    for col, val in valores_imputados_mediana.items()
] + [
    {'Variable': col, 'Tipo': 'Categórica', 'Estrategia': 'Moda', 'Valor Imputado': val}
    for col, val in valores_imputados_moda.items()
])
print('Parámetros calculados y aplicados en la imputación:')
display(df_imputaciones)

### **4.3.4. Paso 17: Construir e interpretar la matriz de correlación**

Construimos la matriz de correlación de Pearson para analizar las relaciones lineales entre la energía total consumida, sus sectores componentes, las condiciones ambientales y los factores del calendario.

In [ ]:
# Selección de variables relevantes para correlación (incluyendo consumo, ambiente y tiempo)
cols_corr = [
    'energia_total_kwh',
    'energia_laboratorios_kwh',
    'energia_oficinas_kwh',
    'energia_salones_kwh',
    'energia_comedor_kwh',
    'energia_auditorios_kwh',
    'potencia_total_kw',
    'ocupacion_pct',
    'temperatura_exterior_c',
    'agua_litros',
    'co2_kg',
    'hora',
    'dia_semana',
    'mes',
    'es_fin_semana',
    'es_festivo'
]

# Matriz de correlación lineal de Pearson
matriz_corr = df[cols_corr].astype(float).corr()

# Visualización mediante Mapa de Calor (Heatmap)
plt.figure(figsize=(13, 10))
sns.heatmap(
    matriz_corr,
    annot=True,
    fmt='.2f',
    cmap='vlag',
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Coeficiente de Correlación de Pearson (r)'},
    annot_kws={'size': 8}
)
plt.title('Matriz de Correlación de Pearson - Sede Duitama', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# Ranking de correlación respecto a la variable objetivo (energia_total_kwh)
corr_objetivo = matriz_corr['energia_total_kwh'].drop('energia_total_kwh').sort_values(ascending=False)
df_corr_rank = pd.DataFrame({
    'Variable': corr_objetivo.index,
    'Correlación con Energia Total (r)': corr_objetivo.values,
    'Magnitud Absoluta |r|': np.abs(corr_objetivo.values)
}).sort_values(by='Magnitud Absoluta |r|', ascending=False).reset_index(drop=True)

print('Jerarquía de Correlación respecto a energia_total_kwh:')
display(df_corr_rank)

#### **Interpretación Detallada de la Matriz de Correlación**

1. **Variables Derivadas y Casi Determinísticas (\(r \approx 0,99 - 1,00\)):**
   - `potencia_total_kw` y `co2_kg` presentan una correlación lineal prácticamente perfecta con la energía total. Esto obedece a razones de modelado físico: la potencia integrada en 1 hora es numéricamente igual a la energía activa, y las emisiones de \(CO_2\) se calculan multiplicando los kWh por un factor fijo de emisión. **Alerta de modelado:** No deben utilizarse como predictores, pues generarían una fuga de datos inmediata.
2. **Variables de Sectores Componentes (\(r \approx 0,65 - 0,78\)):**
   - `energia_laboratorios_kwh`, `energia_oficinas_kwh` y `energia_salones_kwh` presentan correlaciones elevadas con la energía total porque constituyen su suma directa. Si buscamos predecir el total general de la sede sin conocer de antemano el consumo individual, deben excluirse para evitar multicolinealidad estructural.
3. **Variable Operacional Principal (\(r \approx 0,66\)):**
   - `ocupacion_pct` es la variable exógena con mayor capacidad predictiva genuina: a mayor porcentaje de aforo estudiantil y profesoral, mayor encendido de iluminación, computadores y sistemas de climatización.
4. **Variables Temporales y de Calendario:**
   - `hora` (\(r \approx 0,42\)) captura el ciclo diario diurno/nocturno.
   - `es_fin_semana` (\(r \approx -0,26\)) exhibe una correlación negativa coherente: los sábados y domingos cae sensiblemente el consumo institucional.
   - `temperatura_exterior_c` muestra una relación débil a moderada, indicando que el consumo en Duitama depende mucho más de la actividad académica y técnica que de requerimientos severos de calefacción o aire acondicionado.

---
# **Fase IV. Preparación para Modelado**

En la Fase IV consolidamos el conjunto de datos definitivo para alimentar modelos de Machine Learning (e.g., Random Forest, XGBoost, Regresión Regularizada, Redes Neuronales). Se definen la variable objetivo y las predictoras, se excluyen variables contaminantes o redundantes, y se codifican las variables categóricas.

### **4.4.1. Paso 18: Seleccionar las variables objetivo**

Definimos formalmente como **variable objetivo (target)** a predecir:

$$\mathbf{y} = \text{energia\_total\_kwh}$$

**Justificación de Gestión:** Predecir la energía horaria total consumida por la sede Duitama permite a la administración universitaria anticipar la demanda de potencia contratada ante el operador de red, programar alertas tempranas de sobreconsumo y diseñar políticas de descarbonización y eficiencia energética.

### **4.4.2. Paso 19: Definir variables predictoras (features)**

Definimos la matriz de características predictoras \(\mathbf{X}\) agrupando aquellas variables que estarán disponibles legítimamente en el momento de realizar una predicción futura:

- **Temporales / Cíclicas:** `hora`, `dia_semana`, `mes`, `trimestre`, `año`.
- **Calendario Institucional:** `es_fin_semana`, `es_festivo`, `es_semana_parciales`, `es_semana_finales`.
- **Régimen Académico:** `periodo_academico` (categorías: `semestre_1`, `semestre_2`, `vacaciones_mitad`, `vacaciones_fin`).
- **Condiciones de Operación y Ambiente:** `temperatura_exterior_c`, `ocupacion_pct`.

### **4.4.3. Paso 20: Excluir variables no relevantes y con riesgo de fuga de datos**

De acuerdo con el Paso 20 del taller, eliminamos aquellas variables que no aportan poder predictivo generalizable o que incurren en violación de supuestos metodológicos:

| Variable Excluida | Motivo de Exclusión | Riesgo Mitigado |
| :--- | :--- | :--- |
| `reading_id` | Identificador unívoco arbitrario (clave subrogada). | Sobreajuste (*overfitting*) a índices de fila sin valor semántico. |
| `timestamp` | Cadena temporal continua redundante (ya descompuesta en hora, día, mes, año). | Incompatibilidad con matrices numéricas para algoritmos lineales/árboles. |
| `dia_nombre` | Redundancia exacta con `dia_semana` (0 = lunes ... 6 = domingo). | Colinealidad exacta innecesaria. |
| `sede`, `sede_id` | Constantes en el dataset de Duitama (Varianza = 0). | Columnas mono-valor que no aportan varianza explicativa al modelo. |
| `potencia_total_kw` | Medición simultánea de potencia que deriva matemáticamente el total. | **Fuga de datos (Data Leakage):** En producción no se conoce la potencia simultánea si no se conoce la energía. |
| `co2_kg` | Variable consecuencia calculada mediante factor de emisión fijo (\(CO_2 = k \cdot E\)). | Fuga de datos severa. |
| `agua_litros` | Medida simultánea no disponible a priori para planificar demanda eléctrica. | Dependencia de sensores adicionales simultáneos no garantizados. |
| `energia_*_kwh` (5 sectores) | La suma de los 5 sectores compone la energía total. | **Fuga de datos estructural:** Si conoces el consumo de los 5 sectores ya no necesitas predecir el total. |

In [ ]:
# Definición programática de variable objetivo y exclusiones
objetivo = 'energia_total_kwh'

variables_excluidas = [
    'reading_id',
    'timestamp',
    'sede',
    'sede_id',
    'dia_nombre',
    'potencia_total_kw',
    'co2_kg',
    'agua_litros',
    objetivo
] + vars_energia

# Lista de características seleccionadas
features = [c for c in df.columns if c not in variables_excluidas]

print('=== SELECCIÓN DE VARIABLES PARA MODELADO ===')
print(f'Variable Objetivo (Target): {objetivo}')
print(f'Variables Predictoras ({len(features)} features):')
for i, f in enumerate(features, 1):
    print(f'  {i:2d}. {f}')

### **4.4.4. Paso 21: Construir el conjunto final para entrenamiento**

Construimos el dataset final procesando las transformaciones necesarias:
1. **Conversión de booleanas a valores binarios enteros (0 y 1).**
2. **Codificación One-Hot Encoding** para la variable categórica `periodo_academico` (`pd.get_dummies`), generando columnas binarias numéricas independientes.
3. **Ensamblado final de la matriz \(\mathbf{X}\) y el vector \(\mathbf{y}\)** en un único DataFrame estructurado.

In [ ]:
# Extracción de matriz de features
X = df[features].copy()

# 1. Conversión de variables booleanas a enteros binarios (0 / 1)
cols_booleanas = X.select_dtypes(include=['bool']).columns
X[cols_booleanas] = X[cols_booleanas].astype(int)

# 2. Codificación One-Hot para la variable categórica periodo_academico
X = pd.get_dummies(X, columns=['periodo_academico'], prefix='periodo', dtype=int)

# 3. Extracción del vector objetivo y ensamblado del DataFrame final
y = df[objetivo]
df_final = pd.concat([X, y], axis=1)

print(f'Dimensiones del conjunto final: {df_final.shape[0]:,} filas x {df_final.shape[1]} columnas')
display(df_final.head())

In [ ]:
# Verificación exhaustiva de calidad e integridad del conjunto final
print('=== VERIFICACIÓN DE INTEGRIDAD DEL DATASET FINAL ===')
print(f'1. Total de valores nulos presentes        : {df_final.isnull().sum().sum()}')
print(f'2. Valores negativos en variable objetivo   : {(df_final[objetivo] < 0).sum()}')
print(f'3. Tipos de datos no numéricos restantes    : {len(df_final.select_dtypes(exclude=[np.number]).columns)}')
print(f'4. Total de columnas predictoras (features) : {len(df_final.columns) - 1}')
print('\nTipos de datos finales por columna:')
display(pd.DataFrame({'Tipo de Dato': df_final.dtypes}))

### **Estrategia Metodológica de Partición de Datos (Train-Test Split Temporal)**

> [!IMPORTANT]
> **Regla Fundamental para Series de Tiempo:**
> En problemas de pronóstico de consumo energético con dependencia temporal, **NO se debe utilizar partición aleatoria con barajado (`shuffle=True`)**. Barajar los datos introduce fuga temporal hacia el pasado (*look-ahead bias*) y sobreestima irrealmente las métricas de rendimiento.
> La partición debe respetar el orden cronológico estricto: entrenar con años anteriores (e.g., 2018 a 2023) y evaluar en los años más recientes (e.g., 2024 a 2025).

In [ ]:
# Demostración de partición temporal cronológica (80% Entrenamiento - 20% Prueba)
corte_temporal = int(len(df_final) * 0.80)
X_train = df_final.iloc[:corte_temporal].drop(columns=[objetivo])
y_train = df_final.iloc[:corte_temporal][objetivo]
X_test = df_final.iloc[corte_temporal:].drop(columns=[objetivo])
y_test = df_final.iloc[corte_temporal:][objetivo]

print('=== Resumen de Partición Cronológica para Modelado ===')
print(f'Conjunto de Entrenamiento (Train): {X_train.shape[0]:,} observaciones ({X_train.shape[0]/len(df_final)*100:.1f}%)')
print(f'Conjunto de Prueba (Test)        : {X_test.shape[0]:,} observaciones ({X_test.shape[0]/len(df_final)*100:.1f}%)')
print(f'Features de entrada              : {X_train.shape[1]} variables')

### **Exportar el Conjunto de Datos Limpio**

Exportamos el dataset resultante en formato CSV estándar para su reutilización directa en las etapas de entrenamiento de modelos predictivos.

In [ ]:
# Exportar a disco local
nombre_salida = 'dataset_final_duitama.csv'
df_final.to_csv(nombre_salida, index=False)
print(f'✓ Archivo guardado con éxito como: {nombre_salida}')
print(f'  Tamaño del archivo generado: {os.path.getsize(nombre_salida) / (1024**2):.2f} MB')

# Descarga automática si se ejecuta en Google Colab
try:
    from google.colab import files
    files.download(nombre_salida)
    print('✓ Iniciando descarga en navegador local...')
except (ImportError, Exception):
    print('Nota: El archivo se encuentra disponible localmente en el directorio de trabajo.')

---
## **5. Resultado Esperado y Conclusiones Técnicas**

Al culminar el proceso metodológico, se dispone de un conjunto de datos completamente depurado, consistente con las leyes de la física y estructurado para algoritmos de Inteligencia Artificial.

### **Cuadro Comparativo: Estado Inicial vs. Estado Final**

| Métrica / Dimensión | Estado Inicial (Dataset Crudo) | Estado Final (`df_final` Duitama) | Beneficio para Modelado de IA |
| :--- | :---: | :---: | :--- |
| **Registros Totales** | 275.387 (4 sedes mezcladas) | **68.487** (Sede Duitama depurada) | Modelo focalizado en la dinámica específica de Duitama. |
| **Variables / Columnas** | 26 variables mixtas | **16 columnas** (15 features + 1 target) | Espacio vectorial 100% numérico, optimizado y sin redundancias. |
| **Valores Faltantes (Nulos)** | Presentes en 6 variables (hasta 12%) | **0 nulos** (Imputación robusta) | Compatibilidad directa con Scikit-Learn, PyTorch y XGBoost. |
| **Valores Negativos** | 155 registros físicamente absurdos | **0 valores negativos** | Respeta la termodinámica del consumo eléctrico. |
| **Registros Duplicados** | 220 colisiones temporales | **0 duplicados** | Evita sobreponderación de lecturas duplicadas. |
| **Fuga de Datos (Data Leakage)** | Presente en potencia, CO2 y sectores | **Mitigada totalmente** | Predicciones honestas basadas solo en datos disponibles *a priori*. |

### **Recomendaciones para la Fase de Modelado Predictivo**
1. **Modelos Paramétricos y Lineales (Ridge / Lasso):** Recomendable aplicar escalado estándar (`StandardScaler`) en variables numéricas continuas como `temperatura_exterior_c` y `ocupacion_pct`.
2. **Modelos Basados en Árboles (Random Forest / LightGBM / XGBoost):** Altamente recomendados; capturan de manera natural las interacciones no lineales entre las horas del día, las semanas de exámenes y el aforo institucional sin requerir normalización previa.
3. **Modelos de Series Temporales y Redes Recurrentes (LSTM / Temporal Fusion Transformers):** Se sugiere enriquecer las características con rezagos temporales (*lags*, e.g., consumo en \(t-1\), \(t-24\), \(t-168\)) y medias móviles para captar la inercia térmica y operativa.

---
## **6. Instrucciones para la Generación del Archivo de Entrega (PDF)**

Según las especificaciones del Taller (Sección 6 del documento oficial):
> **"Guardar el archivo como PDF y subirlo a la plataforma Moodle."**

Para generar el archivo PDF preservando la totalidad del código, textos explicativos y visualizaciones:

### **Opción 1: Desde Google Colab (Recomendada)**
1. Asegúrate de haber ejecutado todas las celdas en orden (`Entorno de ejecución` -> `Reiniciar y ejecutar todo`).
2. Ve al menú superior: **Archivo (File)** -> **Imprimir (Print)** (o presiona `Ctrl + P` en Windows / `Cmd + P` en Mac).
3. En el cuadro de diálogo de impresión, selecciona como destino: **Guardar como PDF (Save as PDF)**.
4. En la configuración avanzada, marca la casilla **Gráficos de fondo (Background graphics)** para asegurar que los colores de las tablas y gráficos se conserven fielmente.
5. Guarda el archivo con el formato: `Taller_II_Consumo_Energetico_UPTC_NombreEstudiante.pdf` y súbelo a la plataforma Moodle institucional.

### **Opción 2: Desde Jupyter Notebook local (nbconvert)**
Si estás trabajando localmente, puedes exportar desde la terminal ejecutando:
```bash
jupyter nbconvert --to pdf Taller_II_Consumo_Energetico_UPTC.ipynb
```
O mediante el navegador en: **File** -> **Download as** -> **PDF via HTML (.pdf)**.